In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, DataCollatorForLanguageModeling
import transformers
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from datasets import load_dataset, Dataset

In [2]:
app_repo = "/scratch/general/vast/app-repo/huggingface/"
model_name = "Qwen/Qwen3.5-27B" # "google/gemma-4-31B-it" TESTED: google/gemma-3-27b-it   "google/gemma-4-31B-it"
#Needs retraining - "Qwen/Qwen3-32B"
lora_adapter_path = "/uufs/chpc.utah.edu/common/home/u6040150/AI_Programs/lora/lora_" + model_name.split("/")[1]


In [3]:
base_model_path = app_repo + model_name
# Load the base model (you can also load a quantized version here)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    device_map="cuda:0",
    dtype=torch.bfloat16,
    # quantization_config=bnb_config
)

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

In [4]:
# Load the tokenizer from the base model or adapter path (usually the same)
tokenizer = AutoTokenizer.from_pretrained(base_model_path, local_files_only=True) # (base_model_path)

In [5]:
# Attach the LoRA adapter to the base model
model = PeftModel.from_pretrained(base_model, lora_adapter_path)

In [6]:
# Huggingface says this is the way to save the FULL MERGED model
# because libraries like ollama take time to support the newest models.
# Doing this step here saves us time later when we need to convert the model to ollama format.
# tokenizer.save_pretrained("/uufs/chpc.utah.edu/common/home/u6040150/AI_Programs/lora/lora_merged/" + model_name.split('/')[1])
# model = model.merge_and_unload()
# model.save_pretrained("/uufs/chpc.utah.edu/common/home/u6040150/AI_Programs/lora/lora_merged/" + model_name.split('/')[1])

In [7]:
# Some warnings and what they mean:
# "Some parameters are on the meta device because they were offloaded to the cpu."
# Translation: Your model is too big for the GPU but program will try its best and be excruciatingly slow and bad.
# prepare the model input
prompt = "Create a Slurm batch file that can be used to run a Gaussian job with input file AxMeCyHex.com on a CHPC cluster at the University of Utah."
messages = [
  {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
  messages,
  tokenize=False,
  add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completionq
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

content = tokenizer.decode(output_ids, skip_special_tokens=True)

print("content:", content)

content: The user is asking for a Slurm batch script to run a Gaussian computational chemistry job on a CHPC (Center for High Performance Computing) cluster at the University of Utah. I need to create a proper Slurm script with appropriate directives and Gaussian-specific commands.

Key elements needed:
1. Slurm directives (SBATCH commands)
2. Environment setup for Gaussian
3. Job information (name, nodes, tasks, memory, time)
4. Working directory setup
5. Gaussian execution command
6. File handling

For Gaussian on CHPC systems, I'll need to:
- Load the appropriate Gaussian module
- Set up the environment
- Run Gaussian with appropriate parallelization

Let me create a comprehensive template that users can customize.
</think>

# Slurm Batch Script for Gaussian Job on CHPC Cluster (University of Utah)

```bash
#!/bin/bash
#SBATCH --time=8:00:00        # Walltime (HH:MM:SS) - adjust as needed
#SBATCH --nodes=1             # Number of compute nodes
#SBATCH --ntasks=16           # Number 

In [9]:
# Some warnings and what they mean:
# "Some parameters are on the meta device because they were offloaded to the cpu."
# Translation: Your model is too big for the GPU but program will try its best and be excruciatingly slow and bad.
# prepare the model input
prompt = "How can I find what clusters and partitions I have access to at CHPC."
messages = [
  {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
  messages,
  tokenize=False,
  add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

content = tokenizer.decode(output_ids, skip_special_tokens=True)

print("content:", content)

content: The user is asking about finding what clusters and partitions they have access to at CHPC. CHPC likely refers to the Center for High Performance Computing, which exists at several universities (most notably at the University of Utah). This is a common question for users of HPC systems.

To find what clusters and partitions you have access to, users typically need to:

1. Check their allocation status
2. Use SLURM commands to see available partitions
3. Check their group memberships
4. Review documentation or contact CHPC

Let me provide a comprehensive answer about how to check cluster and partition access at CHPC systems.
</think>

To determine which clusters and partitions you have access to at the Center for High Performance Computing (CHPC), follow these steps:

### 1. Check Your Allocation and Access Status
- Visit the **CHPC Portal** at [https://portal.chpc.utah.edu](https://portal.chpc.utah.edu)
- Log in and navigate to the **Allocations** section to view your current a

In [10]:
prompt = "How do I move large data sets in/out of the UofU to my collaborators at a peer university or collaborating entity?"
messages = [  {"role": "user", "content": prompt} ]
text = tokenizer.apply_chat_template(  messages,  tokenize=False,  add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
# conduct text completion
generated_ids = model.generate( **model_inputs, max_new_tokens=32768)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
content = tokenizer.decode(output_ids, skip_special_tokens=True)
print("content:", content)

content: Here's a thinking process that leads to the suggested response:

1.  **Analyze the Request:**
    *   **Topic:** Moving large data sets in/out of the University of Utah (UofU).
    *   **Destination/Source:** Collaborators at a peer university or collaborating entity.
    *   **Context:** University IT policies and services.
    *   **Constraint:** The user is asking for specific institutional guidance (UofU).

2.  **Identify the Authority/Source:**
    *   Since I am an AI, I don't have real-time access to the University of Utah's internal documentation or policy updates.
    *   However, I have access to general knowledge about university data transfer protocols, and I can guide the user on *where* to find the specific, up-to-date information at the UofU.
    *   *Self-Correction/Refinement:* I need to make sure I don't hallucinate specific UofU tools (like "UofU uses Box for everything" - might be true, but could be outdated). I should focus on the *process* and direct them

In [11]:
prompt = "How do I get more storage space for my group?"
messages = [  {"role": "user", "content": prompt} ]
text = tokenizer.apply_chat_template(  messages,  tokenize=False,  add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
# conduct text completion
generated_ids = model.generate( **model_inputs, max_new_tokens=32768)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
content = tokenizer.decode(output_ids, skip_special_tokens=True)
print("content:", content)

content: Here's a thinking process that leads to the suggested response:

1.  **Analyze the Request:**
    *   **Question:** "How do I get more storage space for my group?"
    *   **Context:** This is a request for information about purchasing or allocating storage, likely within an academic or research institution (since "group storage" is a common term in HPC/university IT environments).
    *   **Missing Information:** I don't know *which* institution the user is affiliated with. Storage policies, pricing, and purchasing mechanisms vary wildly between universities, companies, and cloud providers.
    *   **Goal:** Provide a helpful, general answer that guides the user to the specific resources they need, while acknowledging the lack of specific context. I need to sound like an IT support assistant (specifically for CHPC, given the implicit context of my system instructions, though I should be careful not to assume too much if I'm acting as a general AI. *Correction:* Looking at the

In [12]:
prompt = "How do I use my Docker container on the cluster?"
messages = [  {"role": "user", "content": prompt} ]
text = tokenizer.apply_chat_template(  messages,  tokenize=False,  add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
# conduct text completion
generated_ids = model.generate( **model_inputs, max_new_tokens=32768)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
content = tokenizer.decode(output_ids, skip_special_tokens=True)
print("content:", content)

content: The user is asking about how to use Docker containers on a cluster. This is a common question for users working with HPC (High Performance Computing) clusters, as they often need to run containerized applications in batch jobs or on interactive nodes.

Let me provide a comprehensive answer covering the main aspects:

1. Common cluster container runtime options (Singularity/Apptainer, Charliecloud, Docker with root access)
2. How to submit jobs with containers
3. Important considerations (network, storage, permissions)

Since I don't know which specific cluster they're using, I should provide general guidance that applies to most HPC clusters, with emphasis on the most common approaches.

Most HPC clusters don't allow regular Docker due to security and root access requirements. Instead, they typically use:
- Singularity/Apptainer (most common)
- Charliecloud
- Docker in special cases with root access

Let me structure a helpful response.
</think>

# Using Docker Containers on H

In [13]:
prompt = "What is the Protected environment and how do I access it?"
messages = [  {"role": "user", "content": prompt} ]
text = tokenizer.apply_chat_template(  messages,  tokenize=False,  add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
# conduct text completion
generated_ids = model.generate( **model_inputs, max_new_tokens=32768)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
content = tokenizer.decode(output_ids, skip_special_tokens=True)
print("content:", content)

content: Here's a thinking process that leads to the suggested response:

1.  **Analyze the Request:**
    *   **Topic:** Protected Environment (PE) and access methods.
    *   **Context:** The user is asking about a specific system, likely within a university or research institution setting (given the common terminology "Protected Environment" in academia/healthcare).
    *   **Implicit Need:** The user needs to know what it is (definition/purpose) and how to get into it (steps/requirements).
    *   **Constraint:** I am an AI, not the specific IT helpdesk of the user's institution. I don't know *which* institution they belong to (e.g., University of Utah, Harvard, Stanford, etc.). "Protected Environment" is a generic term, though often associated with specific systems like REDCap, EHRs, or HIPAA-compliant servers.
    *   **Observation:** The phrasing "What is the Protected environment" (singular, specific) suggests this might be a copy-paste from a specific institution's FAQ (likely

In [14]:
prompt = "How do I move my data in/out of the Protected Environment?"
messages = [  {"role": "user", "content": prompt} ]
text = tokenizer.apply_chat_template(  messages,  tokenize=False,  add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
# conduct text completion
generated_ids = model.generate( **model_inputs, max_new_tokens=32768)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
content = tokenizer.decode(output_ids, skip_special_tokens=True)
print("content:", content)

content: Here's a thinking process that leads to the suggested answer:

1.  **Analyze the Request:**
    *   **Question:** "How do I move my data in/out of the Protected Environment?"
    *   **Context:** The user is asking about data transfer protocols within a specific IT environment, likely a university or research institution's "Protected Environment" (PE).
    *   **Implicit Need:** The user needs specific, actionable instructions on approved methods for transferring data, adhering to security and compliance policies (like HIPAA, FERPA, etc.).
    *   **Missing Information:** I don't know *which* institution's Protected Environment they are referring to (e.g., University of Utah, University of Washington, Harvard, etc.). Each institution has its own specific tools (Box, Rclone, SFTP, specific portals).
    *   **Constraint:** As an AI, I need to provide general best practices *and* direct them to the specific documentation for their institution, or ask for clarification. However, 

In [15]:
prompt = "What GPUs are available, and how can I access them?"
messages = [  {"role": "user", "content": prompt} ]
text = tokenizer.apply_chat_template(  messages,  tokenize=False,  add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
# conduct text completion
generated_ids = model.generate( **model_inputs, max_new_tokens=32768)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
content = tokenizer.decode(output_ids, skip_special_tokens=True)
print("content:", content)

content: The user is asking about GPUs available on CHPC systems and how to access them. Let me search for information about GPUs on our systems.
</think>

# GPUs at the Center for High Performance Computing - The University of Utah

# GPUs at CHPC The Center For High Performance Computing 155 S 1452 E, RM. 405 SLC, UT 84112-0190 801.585.3791 Stay in Touch Find Us Contact Us Webmaster Quick Links A-Z Index Campus Directory Campus Map Shuttle Tracker CIS UMail @ The U © 2026 The University of Utah Nondiscrimination & Accessibility Disclaimer Privacy Credits & Attributions About the U Media Contacts X Facebook Instagram Youtube Last Updated: 1/22/26


In [16]:
prompt = "Are there any large language models (LLMs) available? If so, how can I access them?"
messages = [  {"role": "user", "content": prompt} ]
text = tokenizer.apply_chat_template(  messages,  tokenize=False,  add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
# conduct text completion
generated_ids = model.generate( **model_inputs, max_new_tokens=32768)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
content = tokenizer.decode(output_ids, skip_special_tokens=True)
print("content:", content)

content: Okay, the user is asking if there are any large language models available and how to access them. Let me start by recalling what I know about LLMs. There are several big ones like GPT-4 from OpenAI, Claude from Anthropic, Gemini from Google, and open-source models like Llama from Meta.

First, I should confirm that yes, there are many LLMs available. Then, the user wants to know how to access them. I need to list the main options. For proprietary models, they usually require API access or web interfaces. For example, OpenAI's models are accessed through their website or API, same with Google's Gemini. Open-source models can be downloaded from platforms like Hugging Face, but that requires more technical know-how.

Wait, the user might not be technical. So I should mention both easy ways (like web interfaces) and more advanced methods (APIs, local installation). Also, note that some models have free tiers, like OpenAI's API with limits, or Hugging Face's free models. But for lo

In [17]:
prompt = "Can I run many independent calculations in parallel?"
messages = [  {"role": "user", "content": prompt} ]
text = tokenizer.apply_chat_template(  messages,  tokenize=False,  add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
# conduct text completion
generated_ids = model.generate( **model_inputs, max_new_tokens=32768)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
content = tokenizer.decode(output_ids, skip_special_tokens=True)
print("content:", content)

content: The user is asking about running multiple independent calculations in parallel. This is a common question in scientific computing and HPC (High-Performance Computing) environments. I should provide a comprehensive answer about the different ways to do this, with appropriate caveats about resource usage.

Let me think about what information would be most helpful:

1. Different methods for parallel execution
2. Important considerations (resource usage, cluster policies)
3. Examples of tools and approaches
4. Warnings about not overloading resources

Since this appears to be related to HPC/cluster usage (given the context of such questions), I should focus on that environment while also mentioning general approaches.
</think>

Yes, you can run many independent calculations in parallel, especially in high-performance computing (HPC) environments. Here are the most effective methods and important considerations:

### 1. Using Slurm Batch Scripts with Job Arrays
Slurm supports job a

In [18]:
prompt = "How can I run Matlab in parallel?"
messages = [  {"role": "user", "content": prompt} ]
text = tokenizer.apply_chat_template(  messages,  tokenize=False,  add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
# conduct text completion
generated_ids = model.generate( **model_inputs, max_new_tokens=32768)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
content = tokenizer.decode(output_ids, skip_special_tokens=True)
print("content:", content)

content: The user is asking about running MATLAB in parallel. This is a common question for MATLAB users who want to speed up their computations. There are several ways to run MATLAB in parallel, and I should cover the main approaches:

1. MATLAB Parallel Server (formerly MATLAB Distributed Computing Server)
2. Parallel Computing Toolbox with local parallel pool
3. Parallel functions like parfor, spmd, parfeval
4. Batch jobs on clusters
5. Other parallelization techniques

Let me provide a comprehensive answer covering the most common and useful approaches.
</think>

Running MATLAB in parallel can significantly speed up computationally intensive tasks. There are several ways to achieve this, depending on your hardware, software setup, and computational needs. Here are the most common and effective approaches:

---

### 1. **Using the Parallel Computing Toolbox (Local Parallel Pool)**

This is the simplest method for parallelizing code on a multi-core machine.

- **Check if Parallel Poo

In [19]:
prompt = "I need the ifort Fortran compiler. How can I access it?"
messages = [  {"role": "user", "content": prompt} ]
text = tokenizer.apply_chat_template(  messages,  tokenize=False,  add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
# conduct text completion
generated_ids = model.generate( **model_inputs, max_new_tokens=32768)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
content = tokenizer.decode(output_ids, skip_special_tokens=True)
print("content:", content)

content: The user is asking about accessing the ifort Fortran compiler. ifort (Intel Fortran Compiler) is a commercial compiler from Intel. To use it, users typically need to:

1. Have access to an Intel Fortran Compiler license (either purchased or through academic/research institution)
2. Install the compiler on their system
3. Load the appropriate modules (on HPC clusters or systems with module management)

Since this appears to be related to a research computing environment (given the context of users asking about compilers), I should provide information about:
- How to check if it's available on their system
- How to load it (typically through modules)
- Any licensing considerations
- Where to find documentation

Let me provide helpful guidance that covers the most common scenarios.
</think>

To access the Intel Fortran compiler (ifort), follow these steps:

1. **Check if it's available on your system**  
   Run the following command in your terminal:
   ```bash
   module spider i